In [1]:
pip install pandas requests beautifulsoup4 lxml matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install selenium

Note: you may need to restart the kernel to use updated packages.


In [14]:
#300 ürünü düzgün çeken kod
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time
import re

options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument(
    "--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 Chrome/120 Safari/537.36"
)

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

wait = WebDriverWait(driver, 20)
urunler = []


def fiyat_bul():
    fiyat_seciciler = [
        "div[data-test-id='default-price'] span",
        "span[data-test-id='price-current-price']",
        "[data-test-id='price-current-price']",
        ".price-value",
        "span[class*='price']",
        "div[class*='price']"
    ]

    for selector in fiyat_seciciler:
        try:
            fiyat = driver.find_element(By.CSS_SELECTOR, selector).text
            if fiyat and "TL" in fiyat:
                return fiyat
        except:
            pass

    try:
        body_text = driver.find_element(By.TAG_NAME, "body").text
        fiyatlar = re.findall(r"\d{1,3}(?:\.\d{3})*,\d{2}\s*TL", body_text)
        if fiyatlar:
            return fiyatlar[0]
    except:
        pass

    return "Bulunamadı"


def yorum_sayisi_bul():
    try:
        time.sleep(2)
        body_text = driver.find_element(By.TAG_NAME, "body").text

        kaliplar = [
            r"(\d+)\s*değerlendirme",
            r"(\d+)\s*Değerlendirme",
            r"(\d+)\s*yorum",
            r"(\d+)\s*Yorum",
            r"Yorumlar\s*\((\d+)\)",
            r"Değerlendirmeler\s*\((\d+)\)",
            r"Ürün Değerlendirmeleri\s*\((\d+)\)"
        ]

        for kalip in kaliplar:
            eslesme = re.search(kalip, body_text, re.IGNORECASE)
            if eslesme:
                return eslesme.group(1)

        return "Bulunamadı"

    except:
        return "Bulunamadı"


def teknik_ozellik_bul():
    teknik_text = ""

    try:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight / 2);")
        time.sleep(2)

        butonlar = driver.find_elements(By.TAG_NAME, "button")

        for buton in butonlar:
            buton_text = buton.text.lower()

            if "özellik" in buton_text or "teknik" in buton_text:
                try:
                    driver.execute_script("arguments[0].click();", buton)
                    time.sleep(2)
                    break
                except:
                    pass
    except:
        pass

    try:
        rows = driver.find_elements(By.CSS_SELECTOR, "table tr")

        for row in rows:
            if row.text.strip():
                teknik_text += row.text.strip() + " | "
    except:
        pass

    try:
        body_text = driver.find_element(By.TAG_NAME, "body").text

        anahtar_kelimeler = [
            "RAM",
            "SSD",
            "HDD",
            "İşlemci",
            "Ekran",
            "Ekran Kartı",
            "İşletim Sistemi",
            "Bellek",
            "Depolama",
            "Çözünürlük"
        ]

        for satir in body_text.split("\n"):
            if any(kelime.lower() in satir.lower() for kelime in anahtar_kelimeler):
                teknik_text += satir.strip() + " | "
    except:
        pass

    if teknik_text.strip():
        return teknik_text.strip()
    else:
        return "Bulunamadı"


try:
    kategori_url = "https://www.hepsiburada.com/laptop-notebook-dizustu-bilgisayarlar-c-98"
    urun_linkleri = []

    for sayfa in range(1, 31):
        url = kategori_url + f"?sayfa={sayfa}"
        driver.get(url)
        time.sleep(5)

        for i in range(5):
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)

        link_elemanlari = driver.find_elements(By.XPATH, "//a[contains(@href,'-p-')]")

        for a in link_elemanlari:
            href = a.get_attribute("href")

            if href and href not in urun_linkleri:
                urun_linkleri.append(href)

        print(f"{sayfa}. sayfa tarandı. Toplam ürün linki: {len(urun_linkleri)}")

        if len(urun_linkleri) >= 300:
            break

    urun_linkleri = urun_linkleri[:300]

    print(f"Toplam işlenecek ürün sayısı: {len(urun_linkleri)}")

    for index, link in enumerate(urun_linkleri, start=1):
        try:
            driver.get(link)
            time.sleep(4)

            try:
                urun_adi = wait.until(
                    EC.presence_of_element_located(
                        (By.CSS_SELECTOR, "h1[data-test-id='title']")
                    )
                ).text
            except:
                urun_adi = "Bulunamadı"

            try:
                marka = driver.find_element(
                    By.CSS_SELECTOR,
                    "a[data-test-id='brand']"
                ).text
            except:
                marka = "Bulunamadı"

            fiyat = fiyat_bul()

            try:
                puan = driver.execute_script("""
                    var el = document.querySelector('span[itemprop="ratingValue"]');
                    return el ? el.innerText : 'Puan yok';
                """)
            except:
                puan = "Puan yok"

            yorum_sayisi = yorum_sayisi_bul()

            try:
                aciklama = driver.execute_script("""
                    var el = document.getElementById('Description');
                    return el ? el.innerText : '';
                """)

                if not aciklama:
                    aciklama = "Açıklama yok"
            except:
                aciklama = "Açıklama yok"

            teknik_ozellikler = teknik_ozellik_bul()

            urunler.append({
                "ürün_adı": urun_adi,
                "marka": marka,
                "fiyat": fiyat,
                "puan": puan,
                "yorum_sayısı": yorum_sayisi,
                "teknik_özellikler": teknik_ozellikler,
                "açıklama": aciklama,
                "ürün_linki": link
            })

            print(f"{index}. ürün çekildi: {urun_adi}")

            if index % 20 == 0:
                df = pd.DataFrame(urunler)
                df.to_excel("ham_laptop_verileri.xlsx", index=False)
                print(f"Ara kayıt yapıldı. Kaydedilen ürün sayısı: {index}")

        except Exception as e:
            print(f"{index}. üründe hata oluştu: {e}")
            continue

finally:
    driver.quit()

df = pd.DataFrame(urunler)
df.to_excel("ham_laptop_verileri_2.xlsx", index=False)

print("Ham veri Excel dosyası oluşturuldu: ham_laptop_verileri.xlsx")
print(f"Toplam çekilen ürün sayısı: {len(urunler)}")

1. sayfa tarandı. Toplam ürün linki: 12
2. sayfa tarandı. Toplam ürün linki: 23
3. sayfa tarandı. Toplam ürün linki: 37
4. sayfa tarandı. Toplam ürün linki: 49
5. sayfa tarandı. Toplam ürün linki: 62
6. sayfa tarandı. Toplam ürün linki: 77
7. sayfa tarandı. Toplam ürün linki: 100
8. sayfa tarandı. Toplam ürün linki: 121
9. sayfa tarandı. Toplam ürün linki: 154
10. sayfa tarandı. Toplam ürün linki: 187
11. sayfa tarandı. Toplam ürün linki: 219
12. sayfa tarandı. Toplam ürün linki: 247
13. sayfa tarandı. Toplam ürün linki: 269
14. sayfa tarandı. Toplam ürün linki: 284
15. sayfa tarandı. Toplam ürün linki: 302
Toplam işlenecek ürün sayısı: 300
1. ürün çekildi: Lenovo Ideapad Slim 3 AMD Ryzen 7 7735HS 16GB 512GB SSD Freedos 15.3" Taşınabilir Bilgisayar 83K70098TR
2. ürün çekildi: Lenovo Ideapad Slim 3 AMD Ryzen 5 7535HS 16GB 512GB SSD Freedos 15.3" Taşınabilir Bilgisayar 83K7009ETR
3. ürün çekildi: Medion Signium 14 S1 MD600032 Intel Core 5 120U 16GB 512GB SSD Freedos 14" 120Hz OLED Dizüst